# Business entity resolution: the data, and the challenges

**Task (Amazon ML Challenge 2026).** For every business in Source 1 (S1), find every record in Source 2 (S2)
and Source 3 (S3) that describes the same business. A business may have zero, one or many matches.

This notebook looks at the raw data before any modelling. Each section ends with a numbered
**challenge**, and the rest of the project answers them:

> **Data.** The dataset belongs to the challenge organisers and is not included in this repository
> (see `data/README.md`). The tables below show a few sample rows only.

The primary challenge here is the scale of the data and to handle it in an instance with 30 GB of memory. The raw files hold ~24M records across train and test, and later stages produce hundreds of millions of candidate pairs, more than pandas can hold in 32 GB. DuckDB runs SQL directly on the TSV files, uses all cores, spills to disk instead of crashing when a query outgrows memory, and keeps every intermediate table in one file, so work survives a kernel restart. Results come back to pandas with .df() for display and plotting.

In [ ]:
import os, re, time
import duckdb, pandas as pd, numpy as np, regex
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 220)
DATA_ROOT = os.environ.get('DATA_ROOT', '/home/ec2-user/SageMaker/data')
WORK      = os.environ.get('EDA_WORK',  '/home/ec2-user/SageMaker/eda_work') 
TRAIN, TEST = f'{DATA_ROOT}/dataset/train', f'{DATA_ROOT}/dataset/test'
os.makedirs(WORK, exist_ok=True)

con = duckdb.connect(f'{WORK}/eda.duckdb')
for setting in ["memory_limit='6GB'", "threads=4", "preserve_insertion_order=false", f"temp_directory='{WORK}/tmp'"]:
    con.execute(f"SET {setting}")
q  = lambda sql, params=None: con.execute(sql, params or []).df()
rd = lambda p: f"read_csv('{p}', delim='\t', header=true, all_varchar=true)"
# A few random rows, but the same ones on every run (fixed seed), so the text below stays true.
SAMPLE_ROWS = lambda t, n=5: q(f"SELECT * FROM (SELECT * FROM {t}) USING SAMPLE reservoir({n} ROWS) REPEATABLE (42)")

t0 = time.time()
FILES = {'s1': 'train/train_source1.tsv', 's2': 'train/train_source2.tsv', 's3': 'train/train_source3.tsv',
         'gt': 'train/train_ground_truth.tsv',
         't1': 'test/test_source1.tsv',   't2': 'test/test_source2.tsv',   't3': 'test/test_source3.tsv'}
for name, rel in FILES.items():
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM {rd(f'{DATA_ROOT}/dataset/{rel}')}")
labels = {'s1': 'train S1', 's2': 'train S2', 's3': 'train S3', 't1': 'test S1', 't2': 'test S2', 't3': 'test S3'}
# One view stacking all six files with a 'src' label, so per-source stats are a single GROUP BY.
# BY NAME lines columns up by name rather than position, in case a file orders them differently.
con.execute("CREATE OR REPLACE VIEW allsrc AS " + " UNION ALL BY NAME ".join(
    f"SELECT '{lab}' AS src, * FROM {t}" for t, lab in labels.items()))
print(f'loaded in {time.time()-t0:.0f}s')

## 1. What the data looks like

Every source has the same four columns: `entity_id`, `business_name`, `business_address`, `country`.
The ground truth lists, for each S1 entity, the comma-separated ids of its S2/S3 matches
(an empty list means the business has no match).

In [ ]:
for t in ['s1', 's2', 's3']:
    display(Markdown(f'**{labels[t]}**')); display(SAMPLE_ROWS(t))
display(Markdown('**train ground truth**')); display(SAMPLE_ROWS('gt'))

In [ ]:
summary = q("""
    SELECT src, COUNT(*) AS records, COUNT(DISTINCT entity_id) AS distinct_ids,
           round(100 * AVG((business_address IS NULL OR trim(business_address) = '')::INT), 2) AS pct_missing_address,
           round(median(length(business_address)), 0) AS median_address_chars
    FROM allsrc GROUP BY src ORDER BY src""")
display(summary)

mix = q("SELECT src, lower(trim(country)) AS country, COUNT(*) AS n FROM allsrc GROUP BY ALL")
mix = mix.pivot(index='src', columns='country', values='n').fillna(0)
display(Markdown('**Country mix (% of records)**')); display((mix.div(mix.sum(axis=1), axis=0) * 100).round(1))

**Observations**
- The records are short. There is one free-text address field, with no separate postcode or
  street-number column, so any number has to be pulled out of the text.
- S2/S3 have some missing addresses. For those records, only the name can be used for matching.
- Training covers the US and India only. **France appears only in the test set** (see section 8).

## 2. One business, many records

In [ ]:
# The ground truth stores all matches of an S1 entity in one comma-separated cell.
# Explode it into one row per (S1, candidate) true pair -- far easier to join and count.
con.execute("""CREATE OR REPLACE TABLE gt_pairs AS
    SELECT source1_entity_id AS s1_id, trim(unnest(string_split(matched_entity_ids, ','))) AS cand_id
    FROM gt WHERE matched_entity_ids IS NOT NULL AND trim(matched_entity_ids) <> ''""")
con.execute("CREATE OR REPLACE TABLE cand AS SELECT 'S2' AS src, * FROM s2 UNION ALL BY NAME SELECT 'S3' AS src, * FROM s3")

def show_entity(s1_id):
    """The S1 record followed by all of its true S2/S3 matches."""
    return q("""SELECT 'S1' AS src, * FROM s1 WHERE entity_id = ?
                UNION ALL BY NAME
                SELECT c.* FROM gt_pairs p JOIN cand c ON c.entity_id = p.cand_id WHERE p.s1_id = ?
                ORDER BY src""", [s1_id, s1_id])

# Pick one business per country that has exactly 4 matches spread over BOTH S2 and S3.
# Ordering by a hash instead of LIMIT alone gives an arbitrary but reproducible example.
for ctry in ['us', 'india']:
    ex = con.execute("""SELECT p.s1_id FROM gt_pairs p JOIN cand c ON c.entity_id = p.cand_id
                        JOIN s1 a ON a.entity_id = p.s1_id WHERE lower(trim(a.country)) = ?
                        GROUP BY p.s1_id HAVING COUNT(*) = 4 AND COUNT(DISTINCT c.src) = 2
                        ORDER BY hash(p.s1_id) LIMIT 1""", [ctry]).fetchone()[0]
    display(Markdown(f'**Example ({ctry}): one S1 business and its 4 true matches**')); display(show_entity(ex))

In [ ]:
dist = q("""SELECT CASE WHEN matched_entity_ids IS NULL OR trim(matched_entity_ids) = '' THEN 0
                        ELSE len(string_split(matched_entity_ids, ',')) END AS n_matches,
                   COUNT(*) AS s1_entities
            FROM gt GROUP BY 1 ORDER BY 1""")
ax = dist.plot.bar(x='n_matches', y='s1_entities', legend=False, figsize=(8, 3), color='#4C72B0')
ax.set_xlabel('true matches per S1 entity'); ax.set_ylabel('S1 entities'); ax.set_title('Match-count distribution (train)')
plt.tight_layout(); plt.show()
n_s1 = dist.s1_entities.sum()
print(f"S1 entities: {n_s1:,} | singletons (0 matches): {dist.s1_entities[0] / n_s1:.2%} | "
      f"mean matches: {(dist.n_matches * dist.s1_entities).sum() / n_s1:.2f}")

# Three structural facts in one pass over the true pairs:
#   - is any S2/S3 record claimed by two different S1 entities?  (exclusivity)
#   - do both sides of a true pair always have the same country?
#   - how many S1 entities have matches in both S2 and S3?
facts = con.execute("""
    WITH pc AS (SELECT p.*, c.src, lower(trim(a.country)) = lower(trim(c.country)) AS same_country
                FROM gt_pairs p JOIN cand c ON c.entity_id = p.cand_id JOIN s1 a ON a.entity_id = p.s1_id)
    SELECT COUNT(*),
           (SELECT COUNT(*) FROM (SELECT cand_id FROM gt_pairs GROUP BY 1 HAVING COUNT(DISTINCT s1_id) > 1)),
           AVG(same_country::INT),
           (SELECT COUNT(*) FROM (SELECT s1_id FROM pc GROUP BY 1 HAVING COUNT(DISTINCT src) = 2))
    FROM pc""").fetchone()
print(f"true pairs: {facts[0]:,}\n"
      f"S2/S3 records matched to MORE than one S1 entity: {facts[1]:,}\n"
      f"true pairs with the same country on both sides: {facts[2]:.4%}\n"
      f"S1 entities with matches in BOTH S2 and S3: {facts[3]:,}")

**Observations and challenges**
- **Challenge 1: this is a set-prediction problem.** The typical S1 business has several matches
  spread over both S2 and S3, while a small share has none. The model has to decide *how many*
  candidates to accept for each entity, including zero. That is why the pipeline ends with a
  per-entity decision rule (`pipeline_v5.py`, step s8) instead of a single global threshold.
- **Challenge 2: matches are exclusive.** No S2/S3 record belongs to two S1 businesses, so the
  final assignment enforces one owner per record (step s9). That's free precision.
- True pairs always agree on country, so candidate generation runs within each country. Country
  is treated as an open set of values, because test contains a country that training does not.

## 3. How similar are true matches?

The next cells take a ~1% sample of S1 entities and all of their true matches, and compare the
two sides with plain token sets.

In [ ]:
# hash(id) % 110 = 7 picks a stable ~1% of S1 entities (about 20K) with ALL of their true pairs,
# small enough to analyse in pandas.
samp = q("""SELECT p.s1_id, a.business_name AS s1_name, a.business_address AS s1_addr, lower(trim(a.country)) AS country,
                   c.src, p.cand_id, c.business_name AS c_name, c.business_address AS c_addr
            FROM gt_pairs p JOIN s1 a ON a.entity_id = p.s1_id JOIN cand c ON c.entity_id = p.cand_id
            WHERE hash(p.s1_id) % 110 = 7""")

TOK   = regex.compile(r'[\p{L}\p{M}\p{N}]+')          # keeps combining marks inside a word (see section 5)
INDIC = regex.compile(r'[\u0900-\u0DFF]')
toks  = lambda s: set(TOK.findall(s.lower())) if isinstance(s, str) else set()
norm  = lambda s: ' '.join(TOK.findall(s.lower())) if isinstance(s, str) else ''
def jac(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0

samp['name_jac'] = [jac(toks(a), toks(b)) for a, b in zip(samp.s1_name, samp.c_name)]
samp['addr_jac'] = [jac(toks(a), toks(b)) for a, b in zip(samp.s1_addr, samp.c_addr)]
samp['c_indic']  = samp.c_name.fillna('').map(lambda s: bool(INDIC.search(s)))
samp['addr_missing'] = samp.c_addr.fillna('').str.strip().eq('')
latin = samp[~samp.c_indic]
has_addr = samp[~samp.addr_missing]
zero = latin[latin.name_jac == 0]      # Latin-script true matches whose names share no word at all

print(f"sample: {samp.s1_id.nunique():,} S1 entities, {len(samp):,} true pairs")
print(f"name-token Jaccard on true pairs: mean {samp.name_jac.mean():.3f}")
print(f"  true pairs sharing ZERO name tokens: {(samp.name_jac == 0).mean():.2%} "
      f"(Latin-script candidates only: {(latin.name_jac == 0).mean():.2%})")
print(f"address exactly equal after normalization: {np.mean([norm(a) == norm(b) for a, b in zip(has_addr.s1_addr, has_addr.c_addr)]):.2%}")
print(f"address-token Jaccard: mean {has_addr.addr_jac.mean():.3f}; zero overlap {(has_addr.addr_jac == 0).mean():.2%}")
print(f"of the zero-name-overlap pairs, address Jaccard >= 0.3 recovers {(zero.addr_jac >= 0.3).mean():.1%}")

fig, axs = plt.subplots(1, 2, figsize=(11, 3))
samp.name_jac.plot.hist(bins=20, ax=axs[0], color='#4C72B0', title='Name-token Jaccard, true pairs')
has_addr.addr_jac.plot.hist(bins=20, ax=axs[1], color='#55A868', title='Address-token Jaccard, true pairs')
plt.tight_layout(); plt.show()

In [ ]:
display(Markdown('**True matches whose names share no token at all** (the address still links them)'))
display(zero.sample(min(8, len(zero)), random_state=0)[['country', 's1_name', 'c_name', 's1_addr', 'c_addr', 'addr_jac']].round(2))
display(Markdown('**True matches with the same address written differently**'))
diff = has_addr[(has_addr.addr_jac.between(0.3, 0.7)) & (has_addr.name_jac >= 0.5)]
display(diff.sample(min(6, len(diff)), random_state=1)[['country', 's1_name', 's1_addr', 'c_addr', 'addr_jac']].round(2))

**Observations and challenges**
- **Challenge 3: names alone miss matches.** A noticeable share of true matches have names that share no
  token: a brand name against a legal name, abbreviations, joined-up words (`TechnoSoft` vs
  `Techno Soft`), or a different script. Blocking on names alone would cap recall well below
  100%, so the address is a first-class blocking field and not only a feature.
- **Challenge 4: addresses are noisy, but their tokens overlap.** Exact address equality almost
  never happens. Token overlap is high, though, which is why blocking and features work on
  rare shared tokens weighted by IDF, and not on exact keys.
- Normalization (accents, `.com` suffixes, leading zeros, concatenated and word-sorted forms)
  recovers a large part of the rest (`pipeline_v5.py`, step s2).

## 4. Same name, different business

In [ ]:
display(Markdown('**Most repeated names in train S1**'))
display(q("SELECT lower(trim(business_name)) AS name, COUNT(*) AS s1_records FROM s1 GROUP BY 1 ORDER BY 2 DESC LIMIT 10"))

# 'Near misses': for the sampled S1 entities, every S2/S3 record with EXACTLY the same name in the
# same country. The LEFT JOIN to the ground truth tells us which of those are real matches.
nm = q("""WITH sa AS (SELECT entity_id, business_name, business_address, lower(trim(country)) AS ctry,
                             lower(trim(business_name)) AS k FROM s1 WHERE hash(entity_id) % 110 = 7),
               ca AS (SELECT entity_id, src, business_address, lower(trim(country)) AS ctry,
                             lower(trim(business_name)) AS k FROM cand)
          SELECT sa.entity_id AS s1_id, sa.business_name AS s1_name, sa.business_address AS s1_addr,
                 ca.src, ca.entity_id AS cand_id, ca.business_address AS c_addr, p.cand_id IS NOT NULL AS is_match
          FROM sa JOIN ca USING (k, ctry)
          LEFT JOIN gt_pairs p ON p.s1_id = sa.entity_id AND p.cand_id = ca.entity_id""")
per = nm.groupby('s1_id').is_match.agg(['sum', 'size'])
n_sampled = con.execute("SELECT COUNT(*) FROM s1 WHERE hash(entity_id) % 110 = 7").fetchone()[0]
print(f"candidates with the IDENTICAL name (same country) as a sampled S1 entity: {len(nm):,}; "
      f"only {nm.is_match.mean():.1%} of them are true matches")
print(f"sampled S1 entities with at least one identical-name NON-match: {(per['size'] > per['sum']).sum() / n_sampled:.1%}")

both = per[(per['sum'] >= 1) & (per['size'] - per['sum'] >= 2)].index
ex = nm[nm.s1_id == sorted(both)[0]] if len(both) else nm.head(0)
display(Markdown('**One S1 name, several records with exactly that name. Only the address separates the real matches:**'))
display(ex.sort_values('is_match', ascending=False)[['s1_name', 's1_addr', 'src', 'c_addr', 'is_match']].head(8))

**Challenge 5: identical names are not proof.** Clinics, chains and generic names
(`Primary Care Group`, `Physical Therapy`, ...) repeat hundreds of times. Under F0.5, where
precision counts double, these look-alikes are the main source of false matches. The model gets
features that describe what does *not* match: the rarest unshared token (IDF), the IDF-weighted
unshared share, number conflicts. It also gets per-entity context (rank, gap to the best
candidate, number of strong name matches).

## 5. Scripts: Indian-language names and a tokenizer trap

In [ ]:
s = 'बेस्ट इंफोटेक लिमिटेड'          # "Best Infotech Limited"
print('stdlib re  \\w+ :', re.findall(r'\w+', s))
print('regex \\p{L}\\p{M}:', TOK.findall(s))

display(Markdown('**Share of S2/S3 records whose name is in an Indian script**'))
display(q(r"""SELECT lower(trim(country)) AS country, src,
                     round(100 * AVG(regexp_matches(COALESCE(business_name, ''), '[\x{0900}-\x{0DFF}]')::INT), 2) AS pct_indic_name
              FROM cand GROUP BY ALL ORDER BY ALL"""))
ind = samp[samp.c_indic & ~samp.s1_name.fillna('').map(lambda x: bool(INDIC.search(x)))]
print(f"true pairs with a Latin S1 name and an Indian-script candidate name: {len(ind):,} "
      f"({len(ind) / len(samp):.1%} of sampled pairs); name Jaccard is 0 for {(ind.name_jac == 0).mean():.1%} of them")
display(ind.sample(min(8, len(ind)), random_state=0)[['s1_name', 'c_name', 's1_addr', 'c_addr']])

**Challenge 6: two scripts for one name.**
- Python's built-in `re` treats Indic vowel signs as non-word characters, so `\w+` breaks each
  word into fragments. Every token feature would then be meaningless for these records. The
  project tokenizes with the `regex` module on `\p{L}\p{M}\p{N}`.
- Even with correct tokens, `लिमिटेड` and `limited` share nothing. External transliteration
  services were not allowed, so the pipeline **mines a dictionary from the training labels**:
  it aligns equal-length Latin and Indian-script names of true pairs token by token, and keeps
  majority mappings (1,498 entries, e.g. `प्राइवेट → private`, `ಕನ್ಸಲ್ಟಿಂಗ್ → consulting`).

## 6. Numbers in the address

In [ ]:
NUM  = regex.compile(r'\d+')
# Numbers in an address, with leading zeros dropped so '007' and '7' count as the same number.
nums = lambda s: {n.lstrip('0') or '0' for n in NUM.findall(s)} if isinstance(s, str) else set()
def num_stats(df, label):
    a, b = df.s1_addr.map(nums), df.c_addr.map(nums)
    m = (a.map(len) > 0) & (b.map(len) > 0)
    share = np.mean([len(x & y) > 0 for x, y in zip(a[m], b[m])])
    same  = np.mean([x == y for x, y in zip(a[m], b[m])])
    return {'pairs': label, 'both addresses have numbers': f'{m.mean():.1%}',
            'share >= 1 number': f'{share:.1%}', 'identical number set': f'{same:.1%}'}
display(pd.DataFrame([num_stats(samp, 'true matches'), num_stats(nm[~nm.is_match], 'identical-name NON-matches')]))

**Challenge 7: numbers separate look-alikes.** Street and unit numbers are among the few signals that
tell two same-name businesses apart. Formatting varies (`0073` vs `73`, `12-B`), so the pipeline
extracts numbers, strips leading zeros, and adds shared-number, number-Jaccard and
number-conflict features. A shared rare number is also a blocking key.

## 7. The metric rewards restraint

Macro F0.5 is computed per S1 entity and then averaged. Precision counts twice as much as recall.
A correct empty prediction for a singleton scores 1, and one false match on a singleton scores 0.

In [ ]:
def f05(pred, true):
    pred, true = set(pred), set(true)
    if not pred and not true: return 1.0
    tp = len(pred & true)
    P = tp / len(pred) if pred else 0.0; R = tp / len(true) if true else 0.0
    return 0.0 if P + R == 0 else 1.25 * P * R / (0.25 * P + R)
cases = [('singleton, predict nothing',         [],                        []),
         ('singleton, one false match',         ['x'],                     []),
         ('3 true, predict 1 correct',          ['a'],                     ['a', 'b', 'c']),
         ('3 true, predict 2 correct',          ['a', 'b'],                ['a', 'b', 'c']),
         ('3 true, predict 3 correct + 2 wrong', ['a', 'b', 'c', 'x', 'y'], ['a', 'b', 'c'])]
display(pd.DataFrame([{'case': c, 'F0.5': round(f05(p, t), 3)} for c, p, t in cases]))

**Challenge 8: decide how many matches to accept for each entity.** Finding every true match and
adding two wrong ones scores lower than returning only two correct ones. A single global
threshold ignores this, so the final rule picks, for each S1 entity, the number of top candidates
k (including 0) that maximizes that entity's *expected* F0.5 under calibrated probabilities.

## 8. France: a country with no training data

In [ ]:
display(q("SELECT src, lower(trim(country)) AS country, COUNT(*) AS records FROM allsrc WHERE src LIKE 'test%' GROUP BY ALL ORDER BY ALL"))
display(Markdown('**French records in test S1**'))
display(SAMPLE_ROWS("t1 WHERE lower(trim(country)) = 'france'"))
display(Markdown('**Most common address words, test S1, by country**'))
# Split every test-S1 address into words, count them per country, keep each country's 12 most
# common non-numeric words. QUALIFY filters on the window function, like HAVING does for GROUP BY.
top_words = q(r"""
    WITH w AS (SELECT lower(trim(country)) AS ctry,
                      unnest(string_split_regex(lower(strip_accents(COALESCE(business_address, ''))), '[^a-z0-9]+')) AS w
               FROM t1),
         c AS (SELECT ctry, w, COUNT(*) AS n FROM w
               WHERE length(w) > 1 AND NOT regexp_matches(w, '^[0-9]+$') GROUP BY ctry, w)
    SELECT ctry, w, n FROM c
    QUALIFY row_number() OVER (PARTITION BY ctry ORDER BY n DESC) <= 12
    ORDER BY ctry, n DESC""")
display(top_words.groupby('ctry').w.apply(lambda s: ', '.join(s)).to_frame('top address words'))

**Challenge 9: generalize to an unseen country.** France makes up a sizeable share of test, but
there are no French labels at all. Its addresses (`rue`, `avenue`, five-digit postcodes) and legal
forms (`SARL`, `SAS`) look nothing like US or Indian ones. Nothing in the pipeline hardcodes a
country: IDF weights are fitted on the corpus being scored, and every feature is
language-agnostic. French accuracy still cannot be measured on validation, and it is the main
reason the leaderboard score (0.934) sits below validation (0.9605). See the limitations in the README.

## Summary: from challenges to design

| # | Challenge | Where it is handled |
|---|---|---|
| 1 | Set prediction: 0 to 11 matches per entity | per-entity expected-F0.5 decision rule (s8) |
| 2 | Each S2/S3 record belongs to at most one S1 | global exclusivity (s9) |
| 3 | True matches with no shared name token | address blocking, normalized and concatenated names, expansion keys (notebook 01, s2, s4) |
| 4 | Same address written differently | rare-token IDF blocking and cosine features (notebook 01, s3, s5) |
| 5 | Identical names on different businesses | unshared-rare-token (IDF) features, per-entity context features (s5) |
| 6 | Indian-script names; `\w+` breaks them | `regex` tokenizer, transliteration dictionary mined from labels (s1) |
| 7 | Numbers separate look-alikes | number extraction and features, rare-number blocking key (s2, s4, s5) |
| 8 | Precision counts double | calibrated probabilities, choose k per entity (s8) |
| 9 | France is unseen | country-agnostic features, IDF fitted on test; open problem |

In [ ]:
con.close()   # the scratch database in EDA_WORK can be deleted now